# Day two — Biodiversity, Land Use and Habitat Change Detection
## *PART I: Philippines Climate Zones, revisited*

Yesterday, we wrapped up with the figures from Beck et al. (2023) that showed us both the current Köppen-Geiger zones and those projected under a future climate change scenario. First things first, re-run the setup cell below if the project is not loaded in your Google Colab environment yet:

In [ ]:
# 1
import os                                                                          # in English:
if not os.path.exists("/content/USLS-Workshop"):                                   # IF workshop isn't downloaded yet:
    !git clone "https://github.com/Cas-Dec/USLS-Workshop.git" /content/USLS-Workshop  # download the workshop
# open the workshop
%cd /content/USLS-Workshop
!pip install -q .                                                                  # install what is needed to do the workshop

#from google.colab import drive
#drive.mount('/content/drive')                                                      # connect to your Google Drive

In [ ]:
# 2
import numpy as np                                                      # anything with numbers
import pandas as pd                                                     # anything with tables
import xarray as xr                                                     # geospatial data
import matplotlib.pyplot as plt                                         # make visuals
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

print("Ready!")

### Day two and we're expert programmers: let's build our own climate zones!

Yesterday you classified **Bacolod alone** by hand-computing four numbers and running them through an `if/elif` chain. Today we generalize that into a real, reusable *function*, finish the decision tree we left incomplete, and run it not just for one city but across the **entire Philippines**, at **three points in time**.

Three files already sit in `../data/processed/`, all of them contain monthly `air_temperature` + `precipitation`, 0.1° grid, Philippines-wide:
- Past: `climate_philippines_1901_1930_mean.nc`
- Present: `climate_philippines_1991_2020_mean.nc`
- Future (SSP4-6.0): `climate_philippines_2071_2099_ssp460_mean.nc`

### New tools for today: `def` functions and `for` loops

Yesterday you wrote a decision tree once, for one city, by hand. Today we want to run the *same* decision tree for **thousands of grid points**, three different points in time. Copy-pasting the same 20 lines thousands of times isn't going to work — we need two new tools:

- **`def` functions** — package a block of code into something you can *name* and *reuse*, with different inputs each time. 👉 https://docs.python.org/3/tutorial/controlflow.html#defining-functions
- **`for` loops** — repeat a block of code automatically, once per item in a list (or grid cell). 👉 https://docs.python.org/3/tutorial/controlflow.html#for-statements

Let's start with functions.

In [ ]:
# =============================================================
# 💡 EXAMPLE — a simple function
# =============================================================

def celsius_to_fahrenheit(temp_c):        # 💡 "def" starts a function; temp_c is its input (a "parameter")
    temp_f = temp_c * 9 / 5 + 32
    return temp_f                         # 💡 "return" sends the result back out

# Now we can call it as many times as we want, with different inputs:
print(celsius_to_fahrenheit(0))
print(celsius_to_fahrenheit(28))
print(celsius_to_fahrenheit(100))

🖊️ **Try for yourself!** Make a simple function that tells you how long your name is:

In [ ]:
def how_long_name(name):
    return ...                      # ✏️ remember: "len(...)" counts the number of characters in a string

print(how_long_name("..."))         # ✏️ try out some names

A function can take several inputs, and a `return` statement always ends the function immediately. Notice `temp_c` only exists *inside* the function — that's its own little workspace.

---

### Step 1 — wrap yesterday's decision tree into a function

Let's reload Bacolod's climatology from yesterday (same files, same numbers) so we have something to test our function on:

In [ ]:
# --- Reload Bacolod's 1991-2020 monthly climatology (same as yesterday) ---
t2m_monthly = xr.open_dataarray("../data/processed/t2m_bacolod_monthly.nc")
tp_monthly  = xr.open_dataarray("../data/processed/tp_bacolod_monthly.nc")

coldest_month_T = float(t2m_monthly.min())
warmest_month_T = float(t2m_monthly.max())
annual_mean_T   = float(t2m_monthly.mean())
annual_precip   = float(tp_monthly.sum())
driest_month_P  = float(tp_monthly.min())

print(f"Coldest month mean temperature : {coldest_month_T:.1f} °C")
print(f"Warmest month mean temperature : {warmest_month_T:.1f} °C")
print(f"Annual mean temperature        : {annual_mean_T:.1f} °C")
print(f"Annual precipitation           : {annual_precip:.0f} mm")
print(f"Driest month precipitation     : {driest_month_P:.0f} mm")

### 🖊️ Your turn: wrap the Group A logic into a function

Below is yesterday's logic for detecting **Group A (tropical)**, now with all three subtypes filled in — including **Am (monsoon)**, using the formula from yesterday's `if/elif/else` example (`driest < 60 and annual >= 25 * (100 - driest)`). Everything that *isn't* Group A is temporarily lumped into `"other"` (we'll fix Groups B/C/D/E in Step 2). Your job is just the *wrapping*: turn it into a function called `classify_koppen` that takes the five numbers we just computed and `return`s the resulting climate code.

**Tips:**
- The function needs 5 parameters, in this order: `coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P`
- End the function with `return group + subtype`

In [ ]:
# ✏️ Call the function "classify_koppen":
def ...(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P):

    # Step 1: main group (only Group A is implemented for now)
    if coldest_month_T >= 18:
        group = "A"
    else:
        group = "other"          # ⚠️ Groups B/C/D/E come in Step 2

    # Step 2: subtype (only defined for Group A)
    if group == "A":
        if driest_month_P >= 60:
            subtype = "f"                                                     # Af — rainforest
        elif driest_month_P < 60 and annual_precip >= 25 * (100 - driest_month_P):
            subtype = "m"                                                     # Am — monsoon
        else:
            subtype = "w"                                                     # Aw — savanna
    else:
        subtype = ""

    ...   # ✏️ return climate code (group + subtype). hint: use the f-string formatting we learned yesterday: f"{...}{...}"


# --- Sanity check: does Bacolod still come out the same as yesterday? ---
bacolod_climate = classify_koppen(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P)
print(f"Bacolod's Köppen-Geiger climate type: {bacolod_climate}")

> 💬 **Discussion:** Does `bacolod_climate` match what you found by hand yesterday (Af)? A function is just a *reusable wrapper* — the science inside hasn't changed yet.

---

### Step 2 — completing the decision tree

Yesterday, Group B (arid) used a placeholder threshold, and Groups C, D, and E weren't implemented at all. Let's finish the job properly, using the actual criteria from [Peel, Finlayson & McMahon (2007)](https://doi.org/10.5194/hess-11-1633-2007) — the paper behind the modern, widely-used version of the Köppen-Geiger map (also what [Beck et al. 2023](https://doi.org/10.1038/s41597-023-02549-6) build on):

| Group | Criterion |
|---|---|
| **A** (Tropical) | coldest month ≥ 18°C |
| **B** (Arid) | annual precipitation below an *aridity threshold* (see below) |
| **C** (Temperate) | coldest month between −3°C and 18°C, warmest month ≥ 10°C |
| **D** (Continental) | coldest month < −3°C, warmest month ≥ 10°C |
| **E** (Polar) | warmest month < 10°C |

**The aridity threshold (Group B)** isn't a single fixed number — it depends on *how a place's rain is distributed across the year*, not just on how much rain falls in total. A place that gets all its rain in summer can "afford" to be drier overall than a place where rain spreads evenly, because summer rain evaporates faster. The real formula (using `annual_mean_T` in °C and `annual_precip` in mm):

- If **≥70%** of annual rain falls Apr–Sep ("summer" in the Northern Hemisphere): `threshold = 20 × annual_mean_T + 280`
- If **≥70%** of annual rain falls Oct–Mar ("winter"): `threshold = 20 × annual_mean_T`
- Otherwise (no strong season): `threshold = 20 × annual_mean_T + 140`
- A place is **Group B** if `annual_precip < threshold`

Since the Philippines sits entirely in the Northern Hemisphere, "summer" always means Apr–Sep here — no extra hemisphere logic needed. Let's see how to compute the % of rain falling Apr–Sep from a 12-month array:

In [ ]:
# =============================================================
# 💡 EXAMPLE — slicing a 12-month array to get Apr-Sep
# =============================================================
monthly_P = tp_monthly.values                # 12 numbers, Jan=index 0, ..., Dec=index 11

# ⚠️ Remember: Python uses zero-based indexing, so Jan=0, Feb=1, ..., Dec=11
summer_months = monthly_P[3:9]               # 💡 slicing: index 3 up to (not including) 9 -> Apr, May, Jun, Jul, Aug, Sep
print("Apr-Sep monthly precip:", summer_months)

summer_precip_pct = summer_months.sum() / monthly_P.sum() * 100
print(f"Percentage of annual rain falling Apr-Sep: {summer_precip_pct:.0f}%")

🖊️ **Try for yourself!** Slice the string below in different ways:

In [ ]:
some_string = "Hey look it's already day two of the workshop!"

first_letter = some_string[...]                 # ✏️ slice the string to get "H"
print(first_letter)

letter_three_to_five = some_string[...:...]     # ✏️ slice the string to get "y l"
print(letter_three_to_five)

# hint: use negative indexing to count from the end of the string
last_letter = some_string[...]                  # ✏️ slice the string to get "!". 
print(last_letter)

day_two = some_string[...:...]                  # ✏️ slice the string to get "day two"
print(day_two)

### 🖊️ Your turn: finish the decision tree

Extend `classify_koppen` to properly determine Groups B, C, D and E. The function now needs a 6th parameter, `summer_precip_pct`, to compute the aridity threshold.

**Tips:**
- Work through the table above in this order: A, then B (arid), then E (polar), then D (continental), then C (temperate) — this order matters, because D and E both need `coldest_month_T < -3`/`warmest_month_T < 10` type checks, and E should "win" over D when both could apply.
- `p_threshold` should be computed *before* the group `if/elif` chain, since Group B's condition depends on it.

In [ ]:
# ✏️ Complete this decision tree:

def classify_koppen(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct):

    # Step 1: aridity threshold (needed for Group B)
    if summer_precip_pct >= 70:
        p_threshold = ...             # ✏️ 20 * annual_mean_T + 280
    elif summer_precip_pct <= 30:
        p_threshold = ...             # ✏️ 20 * annual_mean_T
    else:
        p_threshold = ...             # ✏️ 20 * annual_mean_T + 140

    # Step 2: main group
    if coldest_month_T >= 18:
        group = "A"                          # Tropical
    elif annual_precip < ...:                # ✏️ compare annual_precip against p_threshold
        group = "..."                        # ✏️ Arid
    elif ... < 10:                           # ✏️ which variable decides Polar?
        group = "..."                        # ✏️ Polar
    elif coldest_month_T < ...:              # ✏️ Continental threshold (see table above)
        group = "..."                        # ✏️ Continental
    else:
        group = "..."                        # ✏️ Temperate

    # Step 3: subtype (only defined for Group A) — unchanged from Step 1
    if group == "A":
        if driest_month_P >= 60:
            subtype = "f"                                                     # Af — rainforest
        elif driest_month_P < 60 and annual_precip >= 25 * (100 - driest_month_P):
            subtype = "m"                                                     # Am — monsoon
        else:
            subtype = "w"                                                     # Aw — savanna
    else:
        subtype = ""

    return group + subtype


# --- Sanity check: recompute summer_precip_pct for Bacolod and re-classify ---
summer_precip_pct = tp_monthly.values[3:9].sum() / tp_monthly.values.sum() * 100
bacolod_climate = classify_koppen(coldest_month_T, warmest_month_T, annual_mean_T,
                                   annual_precip, driest_month_P, summer_precip_pct)
print(f"Bacolod's Köppen-Geiger climate type: {bacolod_climate}")

> 💬 **Discussion:** Bacolod is very wet year-round, so it was never at risk of landing in Group B. Are there any parts of the Philippines *do* you expect to be dry enough for Group B?

---

### Step 3 — looping over a handful of cities

`classify_koppen` now works for any location — we just need to *call it repeatedly*. Before jumping to a full 170×135 grid, let's warm up with a `for` loop over a short, hand-written list of a few Philippine cities.

In [ ]:
# =============================================================
# 💡 EXAMPLE — a for loop
# =============================================================
fruits = ["mango", "banana", "durian"]

for fruit in fruits:                 # 💡 "fruit" takes each value in the list, one at a time
    print(f"I like {fruit}!")

💡 Easy, right? You can also loop over pairs of so-called "tuples", 🖊️ **fill in what's missing**:

In [ ]:
# 💡 A list of tuples (pairs):
prices = [("mango", 60), ("...", ...), ("...", ...)]        # ✏️ choose fuits and their prices

# 💡 You can loop over them as well:
for ..., ... in prices:                                     # ✏️ fill in what's missing
    print(f"{name} costs {price} pesos")

💡 Or use `zip` to combine lists into a list of tuples! Look:

In [ ]:
names = ["Alice", "Bob", "Charlie"]
ages = [..., ..., ...]                                      # ✏️ give Alice, Bob and Charlie some ages

# 💡 "zip" combines two lists into a list of tuples
for ..., ... in zip(..., ...):                              # ✏️ fill in what's missing
    print(f"{name} is {age} years old.")

### 🖊️ Classify a handful of Philippine cities

Below is a hand-written list with **approximate** climate numbers (illustrative, not precise data) for a few Philippine cities, in the same order as `classify_koppen`'s parameters — plus Bacolod, using the real numbers you already computed. Loop over the list and print each city's climate code.

In [ ]:
# (city, coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct)
cities = [
    ("Manila",          25.5, 29.5, 27.5, 2000,  15, 85),
    ("Baguio",          15.0, 19.5, 17.5, 3500,  15, 90),
    ("Davao",           26.5, 28.0, 27.2, 1900, 110, 48),
    ("Puerto Princesa", 26.0, 28.5, 27.3, 1450,  45, 72),
    ("Bacolod", coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct),
]

for ..., ..., ..., ..., ..., ... in cities:     # ✏️ loop over cities
    climate_code = ...                          # ✏️ use your classify_koppen()
    print(f"{city}: {climate_code}")        

---

### Step 4 — loading the gridded present-day climate data

Time to go from a handful of hand-typed cities to *every 0.1° grid point in the Philippines* — 170 latitudes × 135 longitudes. This comes from the same [Beck et al. (2023)](https://doi.org/10.1038/s41597-023-02549-6) dataset as yesterday's Köppen-Geiger maps, but here it's the underlying **monthly climate data** (temperature + precipitation) rather than an already-computed classification — we're about to compute the classification ourselves.

In [ ]:
# Load the dataset
ds_present = ...        # ✏️ file: "climate_philippines_1991_2020_mean.nc"

# Inspect it
ds_present

Notice the dimensions: `time` (12 months), `lat` (170), `lon` (135) — this is a genuinely 3D dataset now, unlike yesterday's already-time-averaged maps. `.values` on a variable here gives a `(12, 170, 135)` numpy array: 12 monthly values *for every grid point*.

### 🖊️ Your turn: pull out the two variables as numpy arrays

**Tips:**
- The two variables are `"air_temperature"` and `"precipitation"`
- Use `.values` to get plain numpy arrays (not xarray objects) — easier to index in the loop we're about to write
- Print `.shape` to confirm you get `(12, 170, 135)`

In [ ]:
temp_grid  = ...        # ✏️ variable "air_temperature"
precip_grid = ...       # ✏️ "precipitation"

lats = ...              # ✏️ how did we get coordinate values again?
lons = ...

💡 Use `shape` to inspect the shape of something:

In [ ]:
print("temp_grid shape:  ", temp_grid.shape)

Try it yourself for `precip_grid`:

In [ ]:
...                     # ✏️ print the shape of the precipitation grid

> 💬 **Discussion:** Do you understand what the different values in shape mean for our data? Discuss with your neighbours.

---

### Step 5 — classifying the entire grid

To visit every `(lat, lon)` grid point, we need a loop *inside* a loop — a **nested for loop**. 👉 https://wiki.python.org/moin/ForLoop

💡 **Let's learn some things first**:

1. Here's how you make a 2D array (a table):

In [ ]:
my_array = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print(my_array)

# 💡 this array has 3 rows and 3 columns:
print("my_array shape:", my_array.shape)

2. We can index this array in a bunch of ways, look:

In [ ]:
# get the first row
print(my_array[0])

# that's the same as:
print(my_array[0, :])               # 💡 ":" means "all columns"

# get only the first value of the first row
print(my_array[0, 0])

# get the last row
print(my_array[-1])                 # 💡 remember negative indexing if you want stuff at the end!

# get the first column
print(my_array[:, 0])

# get the first value of the first two rows
print(my_array[0:2, 0])             # 💡 use `:` between two indices as 'to'

# ✏️ get the second column of the last two rows
...

3. Now we can understand the example below: a `for` loop *inside* another `for`loop

In [ ]:
# =============================================================
# 💡 EXAMPLE — a nested for loop
# =============================================================
grid = np.array([[1, 2, 3],
                 [4, 5, 6]])

for i in range(grid.shape[0]):           # 💡 English: make i go from 0 to the number of rows
    for j in range(grid.shape[1]):       # 💡 English: make j go from 0 to the number of columns
        print(f"grid[{i},{j}] = {grid[i, j]}")

One more piece of plumbing before the main exercise: `classify_koppen` returns a compressed code (only `Af`/`Am`/`Aw` are full subtypes — B, C, D and E are left as whole groups). To reuse yesterday's 30-class `KG_INFO` colour scheme for our own map, we pick one representative subtype per group to plot with:

In [ ]:
# Standard Köppen-Geiger color scheme (Beck et al. 2023) — same as yesterday
KG_INFO = [
    (1,  "Af",  "#0000FF"), (2,  "Am",  "#0078FF"), (3,  "Aw",  "#46AAFA"),
    (4,  "BWh", "#FF0000"), (5,  "BWk", "#FF9696"), (6,  "BSh", "#F5A500"),
    (7,  "BSk", "#FFD37F"), (8,  "Csa", "#FFFF00"), (9,  "Csb", "#C8C800"),
    (10, "Csc", "#969600"), (11, "Cwa", "#96FF00"), (12, "Cwb", "#64C800"),
    (13, "Cwc", "#329600"), (14, "Cfa", "#C8FF00"), (15, "Cfb", "#64FF00"),
    (16, "Cfc", "#32C800"), (17, "Dsa", "#FF00FF"), (18, "Dsb", "#C800C8"),
    (19, "Dsc", "#963296"), (20, "Dsd", "#966496"), (21, "Dwa", "#AB00AB"),
    (22, "Dwb", "#780078"), (23, "Dwc", "#4B0082"), (24, "Dwd", "#320050"),
    (25, "Dfa", "#00FFFF"), (26, "Dfb", "#37C8FF"), (27, "Dfc", "#007D7D"),
    (28, "Dfd", "#004B4B"), (29, "ET",  "#B2B2B2"), (30, "EF",  "#686868"),
]
cmap = mcolors.ListedColormap([c for _, _, c in KG_INFO])
norm = mcolors.BoundaryNorm(boundaries=np.arange(0.5, 31.5), ncolors=30)

# Our model only outputs group-level codes for B/C/D/E — map each to one representative subtype:
GROUP_TO_KG_CODE = {
    "Af": 1, "Am": 2, "Aw": 3,
    "B": 6,    # representative: BSh (hot semi-arid) — the most plausible arid subtype in the PH
    "C": 11,   # representative: Cwa (humid subtropical, dry winter)
    "D": 26,   # representative: Dfb — included for completeness, unlikely to appear in the PH
    "E": 29,   # representative: ET — included for completeness, unlikely to appear in the PH
}

### 🖊️ Classify every land pixel

We don't want to classify the ocean, instead, we can just assign it `NaN` (**Not a Number**) values. To do so, we create a grid that we first fill completely with NaNs, and then fill with Köppen-Geiger classification where there actually is land.

💡 Using `np.full`, we can make an array filled with some value we want. Here's how you make a 3x3 grid filled with NaNs by combining np.full with `np.nan`:

In [ ]:
my_empty_grid = np.full((3, 3),     # 3 rows, 3 columns
                        np.nan)     # fill with NaN
my_empty_grid

✏️ **Your turn:** make the Köppen-Geiger grid filled with NaNs.

Hint: a couple of cells above you already extracted `lats` and `lons` from `temp_grid`, use `len` to get the right row (lats) and column (lons) sizes for your grid.

In [ ]:
kg_grid_present = np.full((..., ...),           # ✏️ rows, columns = length of latitudes, length of longitudes
                          ...)                  # ✏️ NaNs

💡 Cool, we have an ampty grid, now let's fill it with values! First, we have to learn one more thing: when our `for` loop comes across an **ocean pixel**, **we want to skip** it. We can do that with `continue`, look:

In [ ]:
brothers = ["Dries", "Bas", "Daan"]

for brother in brothers:
    if brother == "Bas":        # IF Bas
        continue                # -> we skip to next iteration of the loop
    print(brother)

🖊️ **Let's go!** For every grid point that isn't ocean:
1. Pull out its 12 monthly temperature and precipitation values (`temp_grid[:, i, j]` and `precip_grid[:, i, j]`)
2. Compute the same 6 summary numbers as before
3. Call `classify_koppen(...)` and store the numeric KG code in `kg_grid[i, j]` using `GROUP_TO_KG_CODE`

**Tips:**
- `lsm.values[i, j] == 0` means ocean — `continue` skips straight to the next loop iteration

In [ ]:
# 30
lsm = ...                                   # ✏️ "land_sea_mask_0p1.nc". Hint: one variable -> easiest to use `dataarray`

for i in ...(len(lats)):                    # ✏️ how do we make i go from 0 to the number of latitudes?
    for j in ...(len(lons)):                # ✏️
        if lsm.values[i, j] == ...:         # ✏️ what value means ocean in the land-sea mask?
            ...                             # ✏️ how do we skip to the next 'j'?

        monthly_T = ...                     # ✏️ temp_grid[:, i, j]
        monthly_P = ...                     # ✏️

        coldest_month_T   = ...
        warmest_month_T   = ...
        annual_mean_T     = ...
        annual_precip     = ...
        driest_month_P    = ...
        summer_precip_pct = ...             # ✏️ hint: monthly_P[3:9].sum() / monthly_P.sum() * 100

        climate_code = ...                  # ✏️ call classify_koppen(...) with the 6 numbers above
        kg_grid_present[i, j] = GROUP_TO_KG_CODE[climate_code]

print("Done! Unique classes found:", sorted(set(kg_grid_present[~np.isnan(kg_grid_present)])))

---

### Step 6 — map it

Same recipe as yesterday: `pcolormesh` with the `KG_INFO` colormap, and a legend built only from the classes actually present.

In [ ]:
shown_present   = [int(v) for v in sorted(np.unique(kg_grid_present[~np.isnan(kg_grid_present)]))]
patches_present = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in shown_present]

fig, ax = plt.subplots(figsize=(7, 9))
ax.pcolormesh(..., ..., ..., cmap=cmap, norm=norm)     # ✏️ x: lons, y: lats, values: kg_grid_present

# ✏️ mark Bacolod (122.95, 10.68), same as yesterday:
ax.scatter(...)
ax.text(...)

ax.contour(lons, lats, lsm.values, levels=[0.5], colors="black", linewidths=1)

ax.set_title("Our homemade classification — Present (1991-2020)", fontsize=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
fig.legend(handles=patches_present, loc="lower center", ncol=min(len(patches_present), 6),
           frameon=False, title="Köppen-Geiger class", fontsize=9)
fig.suptitle("Homemade Köppen-Geiger Classification — Philippines", fontsize=13)
plt.tight_layout(rect=[0, 0.07, 1, 1])

> 💬 **Discussion:** What do you see? Is Bacolod's own class the one you'd expect from Steps 1-2?

---

### Step 7 — sanity-check against Beck et al.'s own map

How close is our homemade, simplified model to the real thing? Let's plot our map next to `kg_philippines_present.nc` — Beck et al.'s actual 1 km classification from yesterday — side by side.

*(We're only comparing these two maps **by eye** here — a formal pixel-by-pixel agreement score would require regridding one dataset onto the other's resolution, which is a rabbit hole for another day.)*

In [ ]:
# --- Reload Beck et al.'s own present-day classification (same file as yesterday) ---
ds_beck_present = xr.open_dataset("../data/processed/kg_philippines_present.nc")
kg_beck_present = ds_beck_present["kg_class"].squeeze()
kg_beck_present = kg_beck_present.where(kg_beck_present > 0)

shown_beck   = [int(v) for v in sorted(np.unique(kg_beck_present.values)) if not np.isnan(v)]
patches_beck = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in shown_beck]

fig, axes = plt.subplots(1, 2, figsize=(13, 9))

axes[0].pcolormesh(lons, lats, kg_grid_present, cmap=cmap, norm=norm)
axes[0].set_title("Our homemade model (0.1°)", fontsize=12)

axes[1].pcolormesh(kg_beck_present.lon, kg_beck_present.lat, kg_beck_present.values, cmap=cmap, norm=norm)
axes[1].set_title("Beck et al. (2023) (1 km)", fontsize=12)

for ax in axes:
    ax.scatter(122.95, 10.68, color="black", s=40, zorder=5)
    ax.set_xlabel("Longitude")
    ax.set_aspect("equal")
axes[0].set_ylabel("Latitude")

all_shown   = sorted(set(shown_present) | set(shown_beck))
all_patches = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in all_shown]
fig.legend(handles=all_patches, loc="lower center", ncol=min(len(all_patches), 8),
           frameon=False, title="Köppen-Geiger class", fontsize=9)
fig.suptitle("Homemade vs. Beck et al. — Present-day Climate Classification", fontsize=13)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()

> 💬 **Discussion:** Do the two maps roughly agree? Where do they differ most, and why might that be? *(Hint: think about resolution — 0.1° vs. 1 km — and about everything our simplified model leaves out: no seasonality sub-criteria for C/D, no elevation correction, coarser aridity thresholds.)*

---

### Step 8 — past, present, future

We already have the present-day map. Let's repeat the exact same recipe (Steps 4-6) for the **past** (`climate_philippines_1901_1930_mean.nc`) and **future** (`climate_philippines_2071_2099_ssp460_mean.nc`, under the SSP4-6.0 emissions scenario) — same grid, same land-sea mask, same function.

To avoid retyping the whole loop twice, let's wrap Steps 4-5 into one function of our own: `classify_climate_file`, which takes a filename and returns a finished `kg_grid`.

### 🖊️ Writing `classify_climate_file` from scratch

First step: Let's make a function `get_clim_stats` that can read the .nc files with the climate data and returns the statistics that matter for classification.

Before, we made a nested for loop to go over each pixel one-by-one. However, `xarray` methods are designed specifically for geospatial data, and we can get the same results way easier. Typically, xarray methods allow for you to pass a `dim` (dimension) argument. Look:

In [ ]:
ds = xr.open_dataset("../data/processed/climate_philippines_1991_2020_mean.nc")

# 💡 using the `dim` argument in xarray's `mean` method
mean_temperature = ds["air_temperature"].mean(dim="time")

mean_temperature.plot()

**Boom!** Just like that, you computed the mean in time over the entire data array, no need to write a nested for loop that goes over every pixel. **We can even make this plot in a single line of code:**

In [ ]:
xr.open_dataset("../data/processed/climate_philippines_1991_2020_mean.nc")["air_temperature"].mean(dim="time").plot()

**🖊️ Try it yourself:** plot the climatological precipitation `sum` during **1901-1930** in a **single line**:

In [ ]:
# ✏️ in one line: open climate_philippines_1901_1930_mean.nc, compute the precipitation sum over time, and plot it
...

You're really learning to work with `xarray`, congrats! Now you're ready to **🖊️ create** `get_clim_stats`

**💡 Explanation** about the `filename: str = "climate_philippines_1991_2020_mean.nc"` argument below:
1. `str` specifies that our function expects a string to be input
2. the `= "climate_philippines_1991_2020_mean.nc"` makes this file the default, so that if you don't use any argument at all and just run `get_clim_stats`, it will use that file.

In [ ]:
def get_clim_stats(filename: str = "climate_philippines_1991_2020_mean.nc"):
    
    # 1. load the dataset
    ds = xr.open_dataset("../data/processed/" + filename)       # CAS: TEACH THIS!

    # 2. ✏️ extract the temperature and precipitation grids
    temp_grid  = ...
    precip_grid = ...

    # 3. ✏️ compute the climate statistics
    # don't use the float(...) wrappers here, you can't use float() on 2D data
    cm_T = ...      
    wm_T = ...
    am_T = ...
    ap_P = ...
    dm_P = ...
    sp_pct = precip_grid[..., ..., ...].sum(...) / precip_grid.sum(...) * 100 # ✏️ remember your indexing exercises?

    # 4. return the results
    return cm_T, wm_T, am_T, ap_P, dm_P, sp_pct

# if you did it correctly, this should work:
cm_T, wm_T, am_T, ap_P, dm_P, sp_pct = get_clim_stats()
print(cm_T.shape)

If you made that correctly, **these two plots should be the same**:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

cm_T, wm_T, am_T, ap_P, dm_P, sp_pct = get_clim_stats("climate_philippines_1901_1930_mean.nc")
sp_pct.plot(ax=axes[0], cmap="Blues")

da = xr.open_dataset(PROCESSED_DIR / "climate_philippines_1901_1930_mean.nc")["precipitation"]
sp_pct_2 = da[3:9, :, :].sum(dim="time") / da.sum(dim="time") * 100
sp_pct_2.plot(ax=axes[1], cmap="Blues")

**Beast!** Cool, our `get_clim_stats` reads the file we want, and instantly gives us the 2D climate stats needed for Köppen-Geiger classification. 
Step 3 to get to `classify_climate_file`: we need to adapt our `classify_koppen` function so that it can handle the 2D data. Here, we make a new function `classify_koppen_2D`:

In [ ]:
def classify_koppen_2D(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct):
    # 💡 no np.asarray this time - we keep everything as xarray DataArrays, so lat/lon coordinates
    # travel along with the result (needed by the two cells after this one). xarray doesn't support
    # 2D boolean *assignment* directly though, so we mutate each DataArray's underlying .values
    # (a plain numpy array) in place instead - the DataArray wrapper (dims, coords) stays intact.

    p_threshold = xr.where(summer_precip_pct >= 70, 20 * annual_mean_T + 280,
                  xr.where(summer_precip_pct <= 30, 20 * annual_mean_T,
                                                     20 * annual_mean_T + 140))

    # Step 2: main group - same reverse-priority mask assignment as before (least important first,
    # so higher-priority conditions get assigned last and "win"), applied to the .values array
    group = xr.full_like(coldest_month_T, "C", dtype="<U1")   # default/"else" case: Temperate
    group.values[(coldest_month_T < -3).values]       = "D"   # Continental
    group.values[(warmest_month_T < 10).values]        = "E"   # Polar
    group.values[(annual_precip < p_threshold).values] = "B"   # Arid
    group.values[(coldest_month_T >= 18).values]       = "A"   # Tropical (checked first -> assigned last -> wins)
    group.values[np.isnan(coldest_month_T.values)]     = ""    # ocean/no-data - not a real class

    # Step 3: subtype (only defined for Group A) - unchanged logic, just applied to every pixel via masks
    is_A = (group == "A").values
    subtype = xr.full_like(coldest_month_T, "", dtype="<U1")
    subtype.values[is_A & (driest_month_P >= 60).values] = "f"                                                    # Af - rainforest
    subtype.values[is_A & (driest_month_P < 60).values & (annual_precip >= 25 * (100 - driest_month_P)).values] = "m"  # Am - monsoon
    subtype.values[is_A & (driest_month_P < 60).values & (annual_precip < 25 * (100 - driest_month_P)).values]  = "w"  # Aw - savanna

    return group + subtype   # 💡 xarray DataArrays support "+" for element-wise string concatenation too, and keep their lat/lon coordinates

**Epic!** Now we can go straight from file to Köppen-Geiger classification with `classify_climate_file`:

In [ ]:
def classify_climate_file(filename: str = "climate_philippines_1991_2020_mean.nc"):

    # 1. ✏️ use `get_clim_stats` to go from filename to stats
    # Remember: `get_clim_stats` returns 6 values, so we can unpack them into 6 variables
    ..., ..., ..., ..., ..., ... = ...

    # 2. ✏️ use `classify_koppen_2D` to go from the 6 stats to KG classification grid
    kg_grid = classify_koppen_2D(...)

    return kg_grid

If you completed that correctly, this should work:

In [ ]:
# --- Use it for past, present (recomputed as a check) and future ---
kg_grid_past    = classify_climate_file("climate_philippines_1901_1930_mean.nc")
kg_grid_present = classify_climate_file("climate_philippines_1991_2020_mean.nc")
kg_grid_future  = classify_climate_file("climate_philippines_2071_2099_ssp460_mean.nc")

print("Past, present and future grids ready.")

And we can now use your **fully self-built processing pipeline** to visualize climate in the Philippines for 1901-1930, 1991-2020 and 2071-2099 side-by-size:

In [ ]:
# --- Three-panel map: past, present, future ---
def class_to_code(kg_grid):

    # 💡 map each string climate code to its numeric KG_INFO code, keeping the DataArray/coords intact
    code_grid = xr.full_like(cm_T, np.nan, dtype="float64")
    for code, kg_code in GROUP_TO_KG_CODE.items():
        code_grid.values[(kg_grid == code).values] = kg_code

    return code_grid

grids  = [class_to_code(kg_grid) for kg_grid in [kg_grid_past, kg_grid_present, kg_grid_future]]
titles = ["Past\n1901-1930", "Present\n1991-2020", "Future (SSP4-6.0)\n2071-2099"]

all_shown   = sorted(set(int(v) for g in grids for v in np.unique(g.values[~np.isnan(g.values)])))
all_patches = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in all_shown]

fig, axes = plt.subplots(1, 3, figsize=(16, 8))
for ax, grid, title in zip(axes, grids, titles):
    ax.pcolormesh(lons, lats, grid, cmap=cmap, norm=norm)
    ax.scatter(122.95, 10.68, color="black", s=40, zorder=5)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Longitude")
    ax.set_aspect("equal")
axes[0].set_ylabel("Latitude")

fig.legend(handles=all_patches, loc="lower center", ncol=min(len(all_patches), 8),
           frameon=False, title="Köppen-Geiger class", fontsize=9)
fig.suptitle("Homemade Köppen-Geiger Classification — Philippines, Past/Present/Future", fontsize=13)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

### Step 9 — wrap-up: zooming back into Bacolod

Let's check numerically whether our own model predicts Bacolod's climate class changing over time. We need the grid indices closest to Bacolod's coordinates (122.95°E, 10.68°N):

In [ ]:
# 💡 np.argmin(np.abs(...)) finds the index of the closest value in an array
bacolod_i = np.argmin(np.abs(lats - 10.68))
bacolod_j = np.argmin(np.abs(lons - 122.95))

for grid, label in zip(grids, titles):
    code = int(grid[bacolod_i, bacolod_j])
    print(f"{label.splitlines()[0]:9s}: {KG_INFO[code - 1][1]}")

> 💬 **Wrap-up discussion:**
> - What changes across the three maps, broadly? Does the shift match what Beck et al.'s own past→future comparison showed yesterday?
> - Does your homemade model predict a change in Bacolod's own climate class? If so, from what to what — and what would that mean for agriculture, water supply, or biodiversity in Negros?
> - You've now built, from scratch, a full pipeline: raw gridded climate data → a `def` function encoding real scientific criteria → a wrapper function that translates the encoding into results → a map. That's the core pattern behind a huge amount of real climate science.

---

## Well done — Part I complete!

You just:
- Learned `def` functions and `return`
- Learned `for` loops, including nested loops over a 2D grid
- Rewrote yesterday's hand-classification as a reusable, scientifically rigorous function
- Classified the entire Philippines, at three points in time, from raw gridded climate data
- Compared your homemade model against the real Beck et al. (2023) product

Next: **Part II**, where we shift from *climate* to the *life* that climate shapes — vegetation, satellite imagery, and the animals of Negros.

## *PART II: The Philippines as a Biodiversity Hotspot*

### Vegetation types & forest cover

Yesterday and this morning, we classified climate zones using [Köppen's system](https://en.wikipedia.org/wiki/K%C3%B6ppen_climate_classification) — and Köppen, originally a botanist, designed his thresholds specifically so that climate zones would line up with **vegetation zones** (Af regions look like rainforest, Aw regions look like savanna, and so on). In the next part, we look at *actual* vegetation data instead of using climate as a proxy for it.

We'll **zoom into Negros** and use the **[Hansen Global Forest Change dataset](https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11/download.html)** (Hansen et al., 2013, *Science* — 👉 https://doi.org/10.1126/science.1244693), which maps global tree cover and forest loss from satellite imagery at 30 m resolution. You can browse it interactively yourself at 👉 https://glad.earthengine.app/view/global-forest-change.

For now we only need the `treecover2000` variable (percent canopy cover in the year 2000) from `negros_forest_change.nc`. The file is under `../data/processed/day2/`

In [ ]:
# --- Load Negros forest cover data ---
forest_ds = ...   # ✏️ negros_forest_change.nc is in ../data/processed/day2/
forest_ds

**Inspect the data** with a 1x2 (one row, two columns) subplot. Left, you plot `treecover2000`, right, you plot `lossyear`:

In [ ]:
fig, axes = ...                             # ✏️ initiate 1x2 subplot with figsize=(10,5)

forest_ds["treecover2000"].plot(...)        # ✏️ plot the treecover2000 variable from forest_ds, use cmap="Greens" as colormap

...                                         # ✏️ plot the lossyear variable from forest_ds, use cmap="Reds" as colormap

# don't bother with titles or layout, it's just a quick data inspection

> **Discussion:** What do you think the data represents? What is `treecover2000`, and what is `lossyear`? *Hint: the unit is 'years since 2000'*. You might also have noticed that **the figure above took a while to plot**. Why is that, you think?

The file has a very high resolution of 30m. How many data points are you plotting in the point above? **✏️ Calculate it below!**

In [ ]:
# Hint:
# number of data points = two variables x number of latitudes x number of longitudes
nr = ...        # ✏️
nr

💡 If you got that right, you should see that, above, you are plotting **42 240 000 datapoints**, that's a lot. To make our lives a bit easier, let's reduce that **by coarsening it** a bit so maps stay fast to draw. Of course `xarray`&mdash;everyone's favourite library&mdash; has a built-in tool for that:

In [ ]:
treecover = forest_ds["treecover2000"]
treecover_coarse = treecover.coarsen(latitude=10, longitude=10, boundary="trim").mean()     # 💡 what do you think specifying latitude=10, longitude=10 does?
treecover_coarse

**🖊️ How many datapoints does treecover_coarse contain?**

In [ ]:
# Hint: same as before, but only for treecover_coarse
nr = ...        # ✏️
nr

*211 200 datapoints*, good, that's a lot less than before. Let's see if that plots a bit faster:

**🖊️ Plot the tree cover map and mark Bacolod.**

In [ ]:
#  visualize the tree cover data for Negros and mark Bacolod:

fig, ax = ...           # ✏️ initialize fig, ax

...                     # ✏️ plot treecover_coarse

# ✏️ use ax.scatter(...) and ax.text(...) to add a marker and label for Bacolod
# coordinates: 122.95, 10.68
...                
...

ax.set_title("Tree Cover — Negros (year 2000)")
ax.set_aspect("equal")

Way faster!

> **Discussion:** Where on the map is Negros most forested? Does that line up with the wetter (Af/Am) vs. drier (Aw) zones from Part I's homemade climate map?

**🖊️ Summary statistics:** For the entire figure, we can compute the **mean tree cover percentage** simply by using the `.mean()` method. Try it:

In [ ]:
mean_cover = ...            # ✏️ use .mean() on the treecover variable

print(float(mean_cover))

💡 How could you compute the percentage of **pixels with more than 50% tree cover**?

It's super easy, but probably not intuitive for you just yet. Let's guide you through this with an example. Say we take some 2x2 array:

In [ ]:
some_array = np.array([[5, 3], [2, 8]])
some_array

**Where is it > 4?** You can see: first row, first column (number 5); and second row, second column (number 8). **See what this does:**

In [ ]:
some_array > 4

So that gives `True` where the values in some_array are > 4, and `False` where they are not.

In programming, `True` and `False` are often **synonymous** with `1` and `0`, and in this case, we can actually use it just like that, look:

In [ ]:
(some_array > 4).mean()

Two times True (1) and two times False (0) -> the mean is 0.5. So on average, 50% of the values in some_array are > 4. **All we have to do to get a percentage is multiply by 100**:

In [ ]:
my_percentage = (some_array > 4).mean() * 100

print(f"Hey look, {my_percentage}% of the values are greater than 4!")

**🖊️ Your turn: compute what percentage of Negros is forest.** Let's say a pixel with more than 50% tree cover is forest. From the `treecover` DataArray, compute the percentage of the image that meets this criterion:

In [ ]:
percentage_forest = ...           # ✏️ percentage of pixels with treecover > 50%

print(f"Pixels with >50% tree cover: {float(percentage_forest)}%")

> 💬 **Discussion:** Is this what you expected? A mean tree cover of about 12.5% and a mean forest cover of about 14%? It might be quite a bit lower than what you thought. Why? *(Hint: what area are we taking the mean over?)*

Let's take it up one notch and only compute forest cover over land by using the **land sea mask**. The file is called `land-sea-mask_0p00833333.nc` and is under `../data/processed/`:

In [ ]:
lsm = ...    # ✏️ open with xr.open_dataarray()
lsm.plot()

At 1km resolution, the `lsm` DataArray for the Philippines shows where there's land (`True`, or `1`) and where there's ocean (`False` or `0`) — it's a **binary** mask. So now, **how can we use this map to cut out the tree cover data over land only?**

Again, `xarray` has this great built-in function called `.where()` where you can consider a DataArray, e.g. `treecover_coarse`, only where some specific requirement is met.

**For example**, if you're only interested in places where treecover is larger than 50%, you can do:

In [ ]:
treecover_coarse.where(treecover_coarse > 50).plot()

So if we only want to consider the tree cover data where there's land, i.e. where the land sea mask equals one, we should be able to do that with `.where()`, right?

Let's try it naively (a bit of foreshadowing, **this will raise an error**): 

In [ ]:
treecover.where(lsm==1).plot()

💥 **That error is worth reading closely.** `Unable to allocate 508. TiB for an array with shape (4800, 4400, 2040, 1620)...`

> **Discussion:** do you know where those numbers come from?

...

Those are `treecover`'s two dimensions *and* `lsm`'s two dimensions, all four kept separate and multiplied together: 4800 × 4400 × 2040 × 1620 ≈ **5.7 × 10¹³ values**.

There are pobably very little supercomputers in existence that are able to load that into memory, let alone your laptop or the free Google Colab servives.

Why did **xarray** do that instead of just lining the two grids up? Because it **matches dimensions by name**, not by what they represent — and `lsm`'s dimensions are called `lat`/`lon`, while `treecover`'s are called `latitude`/`longitude`. As far as xarray is concerned those are four completely unrelated axes, so it built every combination of every point in both grids instead of comparing them point-for-point.

**💡 The fix: put `lsm` on `treecover`'s exact grid first.** Of coarse, xarray can do that natively. Two steps — `.rename()` to match the dimension names, then `.interp()` to resample `lsm`'s values onto `treecover`'s coordinates.

In [ ]:
# ✏️ Regridding lsm onto treecover's exact grid
lsm_negros = lsm.rename({"lat": "latitude",
                          "...": "..."})        # ✏️ rename lon -> longitude

lsm_negros = lsm_negros.interp(latitude=...,             # ✏️ what latitudes do we want to map lsm to? Hint: treecover.latitude 
                               longitude=...,           # ✏️ what longitudes?
                               method="nearest")

lsm_negros.plot(cmap="Greys_r")

`lsm_negros` now has the exact same shape and dimension names as `treecover` (check: `lsm_negros.shape`). `.where()` can now compare the two point-for-point instead of guessing.

**🖊️ Your turn:** redo the masked plot, using `lsm_negros` instead of `lsm`.

In [ ]:
# ✏️ where do we want to plot the treecover data?
treecover.where(...).plot()   # Hint: where the regridded lsm_negros equals 1

> 💬 **Discussion:** Compare this to the very first tree-cover map you plotted. Did we successfully mask out the ocean pixels using the land sea mask?

Remember the earlier discussion — *"why is the mean tree cover only ~12.5%?"* Now that ocean pixels are properly masked, let's answer it directly.

**🖊️ Your turn:** compute `mean_cover_land`, the mean of `treecover`, masked to land only with `lsm_negros`.

In [ ]:
# Hint: start with treecover -> consider where the regridded lsm equals 1 -> take the mean
mean_cover_land = ...   # ✏️ same .mean() as before, applied to the masked version

print(f"Mean tree cover, ocean excluded: {float(mean_cover_land):.1f}%")

> 💬 **Discussion:** That's noticeably higher than the raw ~12.5% you computed earlier — masking out ocean (which always reads 0% tree cover) was dragging the whole-image average down. General lesson, not just about forests: **always check what your average is actually averaging over.**

---

### Satellite imagery

**A lot of the data we have and use today is thanks to satellites**. What they observe is also **very relevant for a lot of biological applications**. For example, Hansen could never have created the global maps of forest data we used above if it weren't for satellites.

But what does a satellite actually "see"? There's generally two categories of satellites: **active and passive ones:**

![Active vs. passive satellite sensors](images/active-vs-passive-sensors.png)

Passive sensors measure the light that is reflected or emitted by the Earth, whereas active sensors beam light towards the Earth and measure how much is reflected back. For every wavelength of the electromagnetic spectrum (light) they measure, they store data in grids of numbers, very much like the data we have used before.

We'll use the **[Sentinel-2](https://sentinel.esa.int/web/sentinel/missions/sentinel-2)** mission (ESA/Copernicus): two satellites launched in 2015 and 2017 that jointly revisit every point on Earth every ~5 days, at 10-60 m resolution. **🖊️ Open the file** `negros_sentinel2.nc` under `../data/processed/day2/`and see what's inside.

In [ ]:
s2_ds = ...   # ✏️ open the Sentinel-2 data with xr.open_dataset
s2_ds

> **Discussion:** What variables are inside the dataset? What coordinates does it have?

There are **two dates** (`2018-02-09` and `2024-04-03`) and **five variables** available in the dataset.
- **Active measurements**, the satellite beams these to the Earth and measures how much reflects back:
  1. `red`,
  2. `nir` (near-infrared)
- **Passive measurements**, the satellite measures how much of these are reflected by the Earth (coming from the sun):
  1. `visual_r` (red),
  2. `visual_g` (green),
  3. `visual_b` (blue).

The human eye sees (red, green, blue) — if you plot the three passive measurements together you get a ready-made true-colour image (what your eyes would actually see).

**🖊️ Plot the active** `red` **for** `time="2024-04-03"`:

In [ ]:
# ✏️ Load red at 2024-04-03
# Hint: 1) index the "red" variable from s2_ds
# -> 2) use .sel(...) to select the "2024-04-03" timestamp for the time coordinate
red_new = ...

fig, ax = plt.subplots(figsize=(6, 6))
red_new.plot(ax=ax, cmap="Reds")

ax.set_title("Red band only — just one number per pixel")
ax.set_aspect("equal")

> **Discussion:** What do you notice about the red band? What type of surface reflects a lot of red light?

**🖊️ Data quality check:** This is satellite data, and not a ready-made, cleaned dataset. What percentage of values in `red_new` are NaNs?

Hints: remember xarray's `shape`, `np.isnan` and `np.sum`

In [ ]:
# ✏️ Compute the percentage of pixels that has a missing value
# Hint: sum of NaNs / total number of pixels * 100
np.sum(np.isnan(red_new)) / np.prod(red_new.shape) * 100

If you got the above right, you should see that the data is about **0.29% NaNs**&mdash;that's very little. **To get an actual photo-like image**, we need to **combine** the three passive bands (**red, green, blue**) into a single array of shape `(rows, cols, 3)` — one extra dimension holding the 3 colour channels for every pixel. `np.dstack` stacks 2D arrays along a new third axis to do exactly this:

In [ ]:
# =============================================================
# 💡 EXAMPLE — stacking 2D arrays into a 3D array
# =============================================================

array_1 = np.array([[1, 2, 3], [4, 5, 6]])
print(array_1.shape)

array_2 = np.array([[7, 8, 9], [10, 11, 12]])
print(array_2.shape)

# use np.dstack to stack the two arrays, creating a third dimension
array_3 = np.dstack([array_1, array_2])
print(array_3.shape)

> **Discussion:** How many rows and columns do `array_1` and `array_2` have? How many dimensions does `array_3` have?

**🖊️ Try extracting** `array_2` back from `array_3`, this will help you understand the dimensions:

In [ ]:
extracted = array_3[..., ..., ...]   # ✏️ remember how to index?

# if you did that correctly, this should be all be True:
extracted == array_2

**Great!** Now we understand how `np.dstack` works, we can use it to stack our three passive satellite bands! **🖊️ Complete the code below:**

In [ ]:
s2_new = ...    # ✏️ select the 2024-04-03 timestamp from s2_ds. Hint: .sel(...)

# s2_new is a xarray Dataset
# extract the values of the three visual bands:

r = ...         # ✏️ Visual red values. Hint: index the "visual_r" variable from s2_new and get its values with .values
g = ...         # ✏️ Visual green values.
b = ...         # ✏️ Visual blue values.

# 💡 plt.imshow doesn't like NaNs.
# -> we can use np.nan_to_num to replace NaNs with 0s
r = np.nan_to_num(r)
g = np.nan_to_num(g)
b = np.nan_to_num(b)

rgb_new = ...   # ✏️ use np.dstack to stack the three arrays!
print("rgb_new shape:", rgb_new.shape)

# Plotting stuff
rgb_new = rgb_new.astype(np.uint8)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb_new)          # 💡 imshow displays a (rows, cols, 3) array directly as a photo
ax.set_title("True-colour image — Negros, 2024")
ax.axis("off")

> **Discussion:** Cool! A satellite image. Are there any things you notice? Can we observe all of the land surface in this image? 

### 2018 versus 2024, side by side

Build the same kind of true-colour image for the **2018** ("old") time slice, and plot it next to the 2024 image you already have — a two-panel figure, same subplot skills as yesterday.

**Tips:**
- The "old" date is `"2018-02-09"`
- Reuse `rgb_new` (already built above) for the right-hand panel

In [ ]:
# --- Build the "old" (2018) true-colour image ---
s2_old = ...                   # ✏️ pick the 2018 date from s2_ds

# ✏️ extract the values of the three visual bands:
r_old = ...
g_old = ...
b_old = ...

# ✏️ replace NaNs with zeros using np.nan_to_num:
r_old = ...
g_old = ...
b_old = ...

# ✏️ stack the three arrays
rgb_old = ...

# --- Two-panel comparison ---
rgb_old = rgb_old.astype(np.uint8)
fig, axes = plt.subplots(1, 2, figsize=(13, 7))

...         # ✏️ imshow the rgb_old on the first axes
axes[0].set_title("2018")

...         # ✏️ imshow the rgb_new on the second axes
axes[1].set_title("2024")

for ax in axes:
    ax.axis("off")

fig.suptitle("Negros, Sentinel-2 true colour — 2018 vs. 2024")

> 💬 **Discussion:** Looks like 2018 contains a lot more missing data then 2024. But besides that, what visible changes can you spot between 2018 and 2024, just by eye? Can you point out forest, farmland, urban areas, and coastline in the images — and does what you see line up with what you'd expect from the `treecover2000` map, and from Bacolod's known location?

---

### Fauna

Climate and vegetation set the stage — now let's look at who actually lives there. We'll use occurrence records from **[GBIF](https://www.gbif.org/)** (the Global Biodiversity Information Facility), **a free, public database aggregating millions of species sightings** and museum specimens worldwide from thousands of institutions.

Two flagship species first:
- **[Philippine Eagle](https://en.wikipedia.org/wiki/Philippine_eagle)** (*Pithecophaga jefferyi*) — one of the world's largest and rarest eagles, endemic to the Philippines, Critically Endangered, and entirely forest-dependent. 👉 https://www.gbif.org/species/2480381
- **[Large flying fox](https://en.wikipedia.org/wiki/Large_flying_fox)** (*Pteropus vampyrus*) — one of the largest bats in the world, found across the Philippine archipelago. 👉 https://www.gbif.org/species/5218644

![Philippine Eagle](images/philippine-eagle.png)
![Large flying fox](images/large-flying-fox.png)

> 💬 **Discussion:** What visible changes can you spot between 2018 and 2024, just by eye? Can you point out forest, farmland, urban areas, and coastline in the images — and does what you see line up with what you'd expect from the `treecover2000` map, and from Bacolod's known location?

In [ ]:
# --- Load occurrence records ---
eagle = ...     # ✏️ gbif_philippine_eagle.csv under ../data/processed/day2/
flying_fox = ...      # ✏️ gbif_flying_fox.csv, same directory

# ✏️ display first few rows of eagle DataFrame
...

> **Discussion:** What columns do the `eagle` and `flying_fox` data have?

🖊️ How many sightings are there for the two species? You know how to get this by now!

In [ ]:
print(f"Philippine Eagle records: {...}")       # ✏️ print the amount of Philippine Eagle sightings
print(f"Flying fox records: {...}")             # ✏️ print the amount of Flying Fox sightings

### Mapping the sightings

Let's see where these species have been spotted! We can do this easily with `ax.scatter`. However, up until now we've only used `ax.scatter` to mark Bacolod on the map. 

**💡 Quick illustration**:

In [ ]:
lat = [8, 2, 3, 2.5, 7]                 # random values
lon = [120, 125, 130, 135, 140]         # random values

fig, ax = plt.subplots()
ax.scatter(lat, lon)                    # 💡 you can pass arrays to .scatter(...), not just individual values!

🖊️ Let's make a **1x2 subplot** and use `.scatter(...)` to mark the Philippine Eagle observations on the **left** panel, and the Large Flying Fox observations on the **right** panel. We'll also use the 1km resolution `land-sea-mask_0p00833333.nc` from `../data/processed/` to outline the Phillipines.

In [ ]:
...         # ✏️ 1x2 subplot, figsize 12x8

axes[0].scatter(..., ...,             # ✏️ how do we pass the "lon" and "lat" columns from the eagle DataFrame?
                color="firebrick", s=20, alpha=0.7)     

axes[1].scatter(..., ...,   # ✏️ same for the flying_fox
                color="steelblue", s=20, alpha=0.7)     


axes[0].set_title(f"Philippine Eagle ({...} records)")  # ✏️ include the number of records in the title
axes[1].set_title(f"Flying Fox ({...} records)")        # ✏️ include the number of records in the title

lsm = xr.open_dataarray("../data/processed/land-sea-mask_0p00833333.nc")

for ax in axes:
    ax.contourf(..., ..., ...,                   # ✏️ pass the land-sea mask lon, lat, and values to ax.contourf(...) <- makes a filled contour
               levels=[0.75, 1.25], colors=["#666666"],
               linewidths=0.8, alpha=0.2)
    ax.set_xlabel("Longitude")
    ax.set_aspect("equal")

axes[0].set_ylabel("Latitude")

> 💬 **Discussion:** What patterns do you notice in the observation records of the Philippine Eagle and flying fox? The eagle is entirely forest-dependent and Critically Endangered, do you see that reflected in the data?

---

Two species is a start, but let's look at a much richer picture: **every terrestrial mammal record for the Philippines** in GBIF.

The file is `gbif_mammals_philippines.csv` under `../data/processed/day2/`:

In [ ]:
# --- Load the full mammal community dataset ---
mammals = ...    # ✏️ open gbif_mammals_philippines.csv

# ✏️ display first few rows
...

> **Discussion:** How many columns does the `mammals` DataFrame have?

**🖊️ Further inspect the data.** How many total sightings are there? How many unique species are recorded? Hint: you can use pandas' `.nunique()` on the `"species"` column.

In [ ]:
nr_rec = ...               # ✏️ How many total records are there?
print(f"{nr_rec} records.")

# ✏️ How many unique species?
unique_species = ...   # Hint: use .nunique() on the "species" column of the mammals DataFrame
print(f"{unique_species} species")

**🖊️ Which mammal groups dominate?**

Use `.value_counts()` on the `order` column to see which taxonomic groups have the most records.

In [ ]:
order_counts = ...     # ✏️ use .value_counts() on the "order" column of mammals
print(order_counts)

> 💬 **Discussion:** Which order dominates, and by how much? Does that match what you'd naively guess about "Philippine mammals"? *Think about how easy each group is to survey. Record counts reflect sampling effort as much as true abundance!*

---

### Data quality check

Before we filter and plot, let's check how complete the columns we're about to rely on actually are — real-world biodiversity data is rarely 100% populated, and it's good practice to check *before* you filter on a column, not after.

**🖊️ Check for missing values** in the `county`column using `.isnull()`

In [ ]:
# ✏️ Check how many values are missing in the "county" column of the mammals DataFrame
nr_missing = ...          # Hint: use .isnull() to check for missing values, then add .sum() to sum them up

print(f"{nr_missing} missing values")

# ✏️ what percentage of the "county" column is missing?
# 💡 same trick as before: True=1/False=0, so .mean() = fraction missing
perc_missing = ...  # Hint: .isnull(), then .mean(), then multiply by 100 to get percentage
print(f"{perc_missing:.1f}% missing")                   

**🖊️ Check null rates** on the columns we're about to use

We're about to filter records to Negros using `stateProvince`, and later make a bar chart of `iucnRedListCategory`. **Check the missing-value fraction for both**, plus `family` and `order`:

In [ ]:
columns_to_check = ["stateProvince", "iucnRedListCategory", "family", "order"]

# use a for loop to iterate over columns_to_check
for col in columns_to_check:

    # Hint: index mammals with col -> .isnull() -> .mean() -> * 100
    perc_missing = ...     # ✏️  What percentage of the column is missing?

    print(f"{col:20s}: {perc_missing:5.1f}% missing")

> 💬 **Discussion:** `stateProvince` and `iucnRedListCategory` both have a meaningful chunk of missing values, while `family`/`order` are essentially complete. What does that mean for the filtering and bar chart we're about to make?

---

### Filtering to Negros

`stateProvince` isn't perfectly clean text — a look at the raw values shows entries like `"Negros Oriental"`, `"Negros Occidental"`, `"Negros I"`, `"Negros Id."`. Rather than matching one exact string, we can check whether `"Negros"` appears *anywhere* in the text using `.str.contains()`:

In [ ]:
# 7 random strings and a missing value in a pandas Series:
random_strings = pd.Series(["Myles", "Workshop day 2", "Crocodiles", np.nan, "What is a bird?", "Sunday", "One day I'll understand programming", "Sydje is chasing Mellie again"])

# 💡 using .str.contains() to filter rows by a partial text match 
random_strings.str.contains("day",          # -> check if the string "day" is present in each element of the Series
                            case=False,     # -> ignore case. So it doesn't matter if it's Day or DAY or day, it will still match
                            na=False)       # -> treat missing values as "no match"

> **Discussion:** What would the output be if we just looked for the letter "s" instead of for "day"?

`.str.contains()` returns binary `True` / `False` -> *Aha!* Then we can **use it for indexing**!

In [ ]:
contains_str = random_strings.str.contains("...",     # ✏️ toy around with the search string
                                            case=False,     
                                            na=False)
random_strings[contains_str]

**🖊️ Your turn:** filter the mammal records to Negros, and map them

**Tips:**
- Same pattern as the example, applied to `mammals["stateProvince"]`
- Use `mammals[is_negros]` to keep only the matching rows (same boolean-filtering idea as yesterday's `df[condition]`)
- Reuse the scatter + `land_sea_mask.nc` contour pattern from the eagle/flying fox map — but this time, zoom the axes to Negros using `ax.set_xlim(122.3, 123.4)` and `ax.set_ylim(9.8, 11.0)` (the same bounding box as the Sentinel-2/Hansen data)

In [ ]:
is_negros = mammals["stateProvince"].str.contains("...",    # ✏️ what are we looking for? Hint: "Negros"
                                                  case=..., # ✏️ case insensitive
                                                  na=...)   # ✏️ NaNs = no match

negros_mammals = ...    # ✏️ Index mammals with is_negros, as we did above!

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
ax.scatter(negros_mammals["decimalLongitude"], negros_mammals["decimalLatitude"],
           color="purple", s=30, alpha=0.5)
ax.contour(lsm.lon, lsm.lat, lsm.values,
           levels=[0.5], colors=["#666666"], linewidths=0.8, alpha=0.7)

#ax.set_xlim(122.3, 123.4)   # ✏️ FIRST don't set xlim, THEN remove the hashtag
#ax.set_ylim(8.9, 11.0)      # ✏️ FIRST don't set ylim, THEN remove the hashtag

# annotate the locations that occur over 10 times in the data with their total count
counts = (
    negros_mammals.groupby(["decimalLongitude", "decimalLatitude"])
    .size()
    .reset_index(name="count")
)

repeated = counts[counts["count"] > 10]

for _, row in repeated.iterrows():
    ax.scatter(row["decimalLongitude"], row["decimalLatitude"],
               color="black", s=15, zorder=6)
    ax.text(row["decimalLongitude"] + 0.01, row["decimalLatitude"] + 0.01,
            str(int(row["count"])), fontsize=8, color="black", zorder=7)



ax.set_title(f"Mammal occurrence records — Negros (total: {len(negros_mammals)})")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")

> 💬 **Discussion:** Notice the points scattered far outside Negros itself (even outside the Philippines!) if you check `negros_mammals["decimalLongitude"].describe()` — a good reminder that a `stateProvince` text match isn't a perfect filter; some records are simply mislabeled or mis-geocoded. Real data always needs this kind of scrutiny. At some locations, there's more than 100 observations. **What do you think of the geographical distribution of observations in Negros?**

### Conservation status bar chart

Let's check how many animals belong to the different **conservation status categories**! We'll make a bar chart of `iucnRedListCategory` value counts for the Negros mammal subset.

The **[IUCN Red List](https://www.iucnredlist.org/)** categorises extinction risk from least to most severe: **LC** (Least Concern) → **NT** (Near Threatened) → **VU** (Vulnerable) → **EN** (Endangered) → **CR** (Critically Endangered), plus **DD** (Data Deficient — not enough information to assess).

**Tip:**
- `.value_counts()` on `negros_mammals["iucnRedListCategory"]` gives you **counts per category** directly

In [ ]:
iucn_counts = ...    # ✏️ use .value_counts() on the ["iucnRedListCategory"] column of negros_mammals
iucn_counts

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(..., ..., color="darkorange")   # ✏️ x: iucn_counts.index. y: iucn_counts.values

ax.set_title("IUCN Red List Category — Negros mammal records")
ax.set_xlabel("Category")
ax.set_ylabel("Number of records")

> **Discussion:** To what category to most of the observed species belong? Notice that there's quite some in the `DD` category (data deficient), why do you think that is?

**🖊️ Look up** some of the specific `species` behind the EN/CR/VU records on Negros. What species are they? What threatens them?

In [ ]:
# Step 1: binary (True / False) of where the category is EN/CR/VU
is_threatened = ...      # ✏️ Hint: apply .isin(["...", "...", "..."]) to negros_mammals["iucnRedListCategory"]

# Step 2: use your binary to filter negros_mammals
filtered = ...     # ✏️ index negros_mammals with is_threatened, then index to the "species" column

# Step 3: use .unique() to get the unique species names
unique_threatened_species = ...       # ✏️ use .unique() on filtered to get the unique species names

print(unique_threatened_species)

> **Discussion:** Does this match what you already know from your biology studies? What vulnerable, endangered and critically endangered species did we find in this dataset? What do you think are the causes for their conservation status? **Nine species** identified that belong to EN/CR/VU categories in Negros. Do you think that is **accurate?**

**💥 CHALLENGE 💥** Do the same as in the cell above, but in one line of code:

In [ ]:
# [OPTIONAL] ✏️ do what the cell above did, but in one line
print(negros_mammals[negros_mammals["iucnRedListCategory"].isin(["EN", "CR", "VU"])]["species"].unique())

---

### Your own biodiversity map

**Biodiversity** is not such a straightforward concept. Biologists, like yourself, often distinguish between different levels such as **genetic diversity**, **species diversity**, **species abundance**, **ecosystem diversity**, ... With the `mammals` DataFrame, we have a bunch of georeferenced observations. The easiest ways of translating that to biodiveristy are probably species *diversity and abundance*. **Let's go for the former.**

We already worked with the Hansen tree cover data, which is a grid in which the value in each pixel is the percentage of the pixel that is covered by trees. What if we could make something like that ourselves, but for mammal biodiversity? Let's do that: **let's make a grid in which each pixel value is the total number of unique species observed**.

There's a bit of a difficulty: as we saw earlier, if we keep the exact locations and just sum the unique species at every exact location, the biodiversity at one place will be different from another place just 10m away, which is unrealistic. With a small `pandas` trick we can fix that: **by slightly rounding coordinates**, we turn scattered points into a **grid**, and then we can count **unique** values per grid cell.

**💡 Trick #1 — rounding turns points into a grid.** Two GPS coordinates are almost never *exactly* equal, even for animals spotted right next to each other — one extra decimal of precision and they look like different locations. But if we **round** every coordinate to the same number of decimal places, nearby points collapse onto the same rounded value. That shared rounded value becomes a grid cell.

In [ ]:
# =============================================================
# 💡 EXAMPLE — rounding coordinates builds a grid
# =============================================================
toy = pd.DataFrame({
    "species": ["eagle", "eagle", "flying_fox", "tarsier", "flying_fox"],
    "lon":     [122.92,  122.93,  122.94,        124.60,    124.62],            # <- coordinates with 2 decimal points
    "lat":     [10.68,   10.67,   10.66,         9.70,      9.71],
})

toy["lon_grid"] = toy["lon"].round(1)   # <- make a new column "lon_grid" that is a copy of "lon" but rounded to 1 decimal value
toy["lat_grid"] = toy["lat"].round(1)   # <- same for "lat"
toy

> Discussion: Look at the `lon` column versus the `lon_grid` column. Do you see how the first three observations, which have different exact locations, now share the same grid point?

**Rounding to 1 decimal** place groups points within roughly **~10 km** of each other into the same cell.

**Let's apply the same trick to the real `mammals` table!**

First, a data-quality step: a handful of records sit way outside the Philippines. Let's filter those out. By now you probably already know the drill: first you make some binary condition, then you filter the DataFrame rows using that condition. To cut `mammals` to the Philippines, we need to filter both `lon` and `lat`.

Time for you to learn something new: filtering two conditions at once using `&`, and using `.between()` to allow a range of values:

In [ ]:
# 💡 .between() to allow a range of values
# 💡 & means 'AND' -> you have to meet condition 1 AND condition 2
condition = toy["lon_grid"].between(122.8, 123.0) & toy["lat_grid"].between(10, 11)
print(condition)

toy[condition]

**🖊️ Your turn:** Keep only records inside a generous Philippines bounding box before gridding, using `.between()` and `&` to filter both `decimalLongitude` and `decimalLatitude` at the same time:
- Philippines bounding box: longitude **116–127°E**, latitude **4.5–21.5°N**
- Round `decimalLongitude`/`decimalLatitude` to 1 decimal place, same as the toy example, into new columns named `lon_grid` / `lat_grid`

In [ ]:
# ✏️ filter mammals both on decimalLongitude and decimalLatitude
in_philippines = ...   # ✏️ Look at the example above! Double condition

mammals_ph = ...       # ✏️ use in_philippines to index mammals

print(f"{len(mammals)} records -> {len(mammals_ph)} after dropping out-of-bounds points")

# Same rounding trick as the toy example above
mammals_ph["lon_grid"] = ...   # ✏️ new column "lon_grid" that is a copy of "decimalLongitude" but rounded to 1 decimal value

mammals_ph["lat_grid"] = ...    # ✏️ same for "lat_grid" and "decimalLatitude"

mammals_ph.head()

**💡 Trick #2 — counting *unique* species per grid cell.**

You've already used `.value_counts()` to count *records*. What we want now is **subtly different**: per grid cell, how many *different* species were seen — a cell with 3 records of the same species shouldn't score higher than a cell with 2 records of 2 different species.

`.pivot_table()` builds exactly this kind of grid in one call: a row grouping, a column grouping, a value to summarize, and how to summarize it — here, `"nunique"`, short for *number of unique values*:

In [ ]:
# =============================================================
# 💡 EXAMPLE — pivot_table: row grouping x column grouping -> one summary value
# =============================================================
toy_richness = toy.pivot_table(index="lat_grid",        # -> group by the "lat_grid" column
                               columns="lon_grid",      # -> group by the "lon_grid" column
                               values="species",        # -> we want to aggregate the "species" column
                               aggfunc="nunique")       # -> how? With the number of unique species in each lat_grid x lon_grid combination
toy_richness

Both cells come out to **2** — cell `(10.7, 122.9)` had 3 records but only 2 distinct species (eagle, eagle, flying_fox); cell `(9.7, 124.6)` had 2 records and 2 distinct species (tarsier, flying_fox). `pivot_table` + `"nunique"` counted *species*, not *sightings*.

**🖊️ Your turn:** build the real richness grid — same call, real data.

- Same `pivot_table()` shape as above: `index="lat_grid"`, `columns="lon_grid"`, `values="species"`, `aggfunc="nunique"`
- Apply it to `mammals_ph`, and call the result `richness_ph`

In [ ]:
biodiversity = mammals_ph.pivot_table(index="lat_grid",      # ✏️ use pivot_table on mammals_ph!
                                     columns="lon_grid",   
                                     values="species",
                                     aggfunc="nunique")

print(f"Grid shape: {biodiversity.shape}, richest cell: {int(biodiversity.max().max())} species")
biodiversity

> **Discussion:** Richest 10x10km pixel has a total count of 47 unique mammal species. Do you think that is representative of the biodiversity in the Philippines?

**💡 One more ingredient — the same forest-cover data, for the whole country.**

You already know how to load and coarsen Hansen tree-cover data — you did exactly this for Negros, at the very start of Part II. `forest_cover_philippines_1km.nc` is the same dataset and variable, just for the whole archipelago. Two differences to watch for: its coordinates are named `lat`/`lon` here instead of `latitude`/`longitude`, and its native pixel size is different, so the coarsening factor needs adjusting to land on roughly the same ~0.1° cells as `biodiversity` (native pixels are ~0.0083°, so ~12 of them make up ~0.1°).

Load it, coarsen it, then we'll put both maps side by side.

In [ ]:
treecover_ph_ds = ...   # ✏️ forest_cover_philippines_1km.nc in ../data/processed/
treecover_ph = ...      # ✏️ select the "treecover2000" variable from treecover_ph_ds

# ✏️ same .coarsen() call as treecover_coarse earlier
treecover_ph_coarse = treecover_ph.coarsen(lat=...,      # ✏️ coarsen lat 12x
                                           lon=...,      # ✏️ coarsen lon 12x
                                           boundary="trim").mean()
treecover_ph_coarse

In [ ]:
# --- Two-panel comparison: species richness vs. tree cover, whole Philippines ---
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

# 💡 given — pivot_table's output isn't an xarray DataArray, so we plot it with pcolormesh instead of .plot()
mesh = axes[0].pcolormesh(biodiversity.columns.values, biodiversity.index.values, biodiversity.values,
                           cmap="viridis", shading="nearest")
fig.colorbar(mesh, ax=axes[0], label="Unique species recorded", shrink=0.7)
axes[0].set_title("Mammal species richness (GBIF)")

# ✏️ On axes[1], plot treecover_ph_coarse where treecover_ph_coarse > 0
# Use a green colormap: cmap="Greens"
...

axes[1].set_title("Tree cover (Hansen)")

for ax in axes:
    # plot lsm contourf
    ax.contourf(lsm.lon, lsm.lat, lsm.values,
                levels=[0.75, 1.25], colors=["#666666"],
                linewidths=0.8, alpha=0.2)
    ax.set_xlim(116, 127)
    ax.set_ylim(4.5, 21.5)
    ax.set_aspect("equal")

> 💬 **Discussion:** Where does the richness map light up brightest — does it track the tree-cover map, or does it look more like "wherever a biologist happened to visit"? Compare the hotspots to the sampling-bias pattern you already noticed in the eagle/flying-fox maps earlier in this Part. **A high-richness cell can mean real biodiversity, or it can just mean someone was there recording things** — with presence-only data like this, the map alone can't always tell you which.

---

## Well done — Part II complete!

You just:
- Connected yesterday's climate classification to actual vegetation and forest cover data
- Learned to build true-colour images from raw satellite bands (`np.dstack`)
- Compared satellite imagery across time, by eye
- Worked with real, messy biodiversity occurrence data (GBIF): filtering, null-checking, and conservation-status reporting
- Built your own species-richness grid from scratch (`round` + `pivot_table`) and used it to ask whether biodiversity tracks forest cover across the whole country

Next: **Part III**, where we go from satellite imagery to an actual land use classification.

---

## *PART III: From satellite imagery to land use*

Köppen designed his climate thresholds around vegetation zones; now let's flip that around and classify *land cover itself* directly from the Sentinel-2 imagery you already loaded in Part II — using the exact same recipe as Part I's homemade climate classifier: turn a decision rule into a `def` function, apply it to a whole grid at once, and map the result.

### 1) NDVI: turning two bands into one meaningful number

**Healthy, lush-green vegetation absorbs a lot of red light but reflects a lot of near-infrared**. Remember that we had these two bands available in our Sentinel-2 imagery? We can combine them in the [Normalized Difference Vegetation Index NDVI](https://en.wikipedia.org/wiki/Normalized_difference_vegetation_index), the most widely used metric to derive vegetation status from remote sensing data. NDVI is computed as:

**NDVI = (NIR − Red) / (NIR + Red)**

NDVI ranges from -1 to +1: strongly **negative** for water, **near zero** for bare soil/urban surfaces, and **high** (often >0.3) for healthy vegetation. Let's write a small reusable function for it:

In [ ]:
def compute_ndvi(red, nir):
    return (nir - red) / (nir + red)


# --- Apply it to the 2024 scene (s2_new, from Part II) and map it ---
ndvi_new = compute_ndvi(s2_new["red"], s2_new["nir"])

fig, ax = plt.subplots(figsize=(7, 7))
ndvi_new.plot(ax=ax, cmap="RdYlGn", vmin=-1, vmax=1, cbar_kwargs={"label": "NDVI"})
ax.set_title("NDVI — Negros, 2024")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Which parts of the map are reddish (low/negative NDVI)? Which are green (high NDVI)? Do the reddish areas match what you'd expect for water and urban areas from the true-colour image earlier?

---

### 2) From NDVI to land-cover classes

Let's turn NDVI into actual land-cover classes using a simple, interpretable rule — deliberately similar in spirit to Köppen's group logic: a handful of threshold conditions on physically meaningful numbers.

Looking at real values in this scene: NDVI ranges roughly -0.95 to +0.97 (mean ~0.32 in 2018, ~0.26 in 2024), and *brightness* (the average of red and nir) is much lower over water than over land or bare ground. That gives us three classes:

| Class | Rule |
|---|---|
| **Water** | low brightness *and* low NDVI (dark in both bands — water absorbs rather than reflects) |
| **Vegetation** | high NDVI |
| **Bare/Urban** | everything else (moderate-to-high brightness, low NDVI) |

We'll use `brightness < 800` and `NDVI < 0.1` for Water, and `NDVI > 0.3` for Vegetation — thresholds picked by eye against this scene's own numbers.

### 🖊️ Your turn: `classify_landcover_2D`

Reuse the exact "boolean-mask, reverse-priority overwrite" trick from `classify_koppen_2D` in Part I — same technique, brand new domain, xarray structure preserved throughout.

In [ ]:
def classify_landcover_2D(red, nir):
    
    ndvi = ...         # ✏️ reuse the function you just wrote
    brightness = (red + nir) / 2

    # 1. We make a map full of "Bare/Urban"
    landcover = xr.full_like(red, "Bare/Urban", dtype="<U10")
    
    # 2. Where it should be Water / Vegetation, we override the "Bare/Urban"
    landcover.values[(... < 800).values & (... < 0.1).values] = "Water"        # ✏️ what should be < 800 and what should be < 0.1?
    
    landcover.values[(...).values] = "Vegetation"                              # ✏️ What's the condition for vegetation?
    
    # red value = NaN = data gap -> not a real class
    landcover.values[np.isnan(red.values)] = ""

    return landcover


# --- Try it on the 2024 scene ---
landcover_new = classify_landcover_2D(s2_new["red"], s2_new["nir"])
print(dict(zip(*np.unique(landcover_new.values, return_counts=True))))

In [ ]:
LC_INFO = [
    (1, "Water",      "#3B4CC0"),
    (2, "Vegetation", "#2E8B57"),
    (3, "Bare/Urban", "#C2A878"),
]
lc_cmap = mcolors.ListedColormap([c for _, _, c in LC_INFO])
lc_norm = mcolors.BoundaryNorm(boundaries=np.arange(0.5, len(LC_INFO) + 1.5), ncolors=len(LC_INFO))
LC_TO_CODE = {name: code for code, name, _ in LC_INFO}


def landcover_to_code(landcover):
    # 💡 same trick as class_to_code in Part I: map each string label to its numeric code, keep the DataArray/coords intact
    code_grid = xr.full_like(landcover, np.nan, dtype="float64")
    for name, code in LC_TO_CODE.items():
        code_grid.values[(landcover == name).values] = code
    return code_grid


fig, ax = plt.subplots(figsize=(7, 7))
code_new = landcover_to_code(landcover_new)
ax.pcolormesh(code_new.longitude, code_new.latitude, code_new, cmap=lc_cmap, norm=lc_norm)
ax.scatter(122.95, 10.68, color="black", s=40, zorder=5)
ax.set_title("Homemade land cover — Negros, 2024")
ax.set_aspect("equal")
patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
ax.legend(handles=patches, loc="lower left", frameon=True, fontsize=9)
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Compare this to the true-colour image from Part II — does Water/Vegetation/Bare-Urban line up with what you saw by eye? Is our map a good map?

---

### 3) Wrap into one reusable function

Same shape as Part I's `classify_climate_file`: load a file, compute what's needed, classify, return.

In [ ]:
def classify_negros_landcover(timestamp="2024-04-03"):
    
    # takes a Sentinel-2 date, returns a (latitude, longitude) DataArray of land-cover labels
    
    s2 = ...    # ✏️ open "negros_sentinel2.nc" from "../data/processed/day2/"
    s2 = ...    # ✏️ Hint: `.sel(time=...)` method. Select the timestamp that is passed to the function

    red = ...   # ✏️ select the "red" variable from s2
    nir = ...   # ✏️ select the "nir" variable from s2
    
    return classify_landcover_2D(..., ...)      # ✏️ use your function!


# --- Use it for both available dates ---
landcover_2018 = classify_negros_landcover("2018-02-09")
landcover_2024 = classify_negros_landcover("2024-04-03")

print("2018:", dict(zip(*np.unique(landcover_2018.values, return_counts=True))))
print("2024:", dict(zip(*np.unique(landcover_2024.values, return_counts=True))))

Map both side by side:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

for ax, landcover, title in zip(axes, [landcover_2018, landcover_2024], ["2018", "2024"]):
    code_grid = landcover_to_code(landcover)
    ax.pcolormesh(code_grid.longitude, code_grid.latitude, code_grid, cmap=lc_cmap, norm=lc_norm)
    ax.scatter(122.95, 10.68, color="black", s=30, zorder=5)
    ax.set_title(title)
    ax.set_aspect("equal")

patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
fig.legend(handles=patches, loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Homemade Land Cover — Negros, 2018 vs. 2024")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

> **Discussion:** What do you notice? Do you think our data is appropriate to accurately visualize land use classes? Is the Bare-Urban class reliable? **What's going wrong?**

Let's sanity-check our 2024 classification against the real `treecover2000` map from Part II — side by side, purely by eye (different sensor, different resolution, so no pixel-by-pixel score, same design choice as Part I Step 7):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

code_2024 = landcover_to_code(landcover_2024)
axes[0].pcolormesh(code_2024.longitude, code_2024.latitude, code_2024, cmap=lc_cmap, norm=lc_norm)
axes[0].set_title("Our homemade land cover (2024)")

treecover.plot(ax=axes[1], cmap="Greens", vmin=0, vmax=100, add_colorbar=False)
axes[1].set_title("Hansen tree cover (2000)")

for ax in axes:
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

> **Discussion:** As you can see, it's not all that straightforward to make a tree cover map as beautiful as that of Hansen. Although we're doing something else - land use classification - *working with satellite data is clearly a tricky business.*

---

## Well done — Part III complete!

You just:
- Learned what NDVI is and why it works (red absorption vs. NIR reflection in healthy vegetation)
- Reused Part I's exact "rule -> vectorized function -> map" pattern in a brand new domain
- Classified real satellite imagery into land-cover classes, for two points in time
- Wrapped the whole pipeline into one reusable function, ready to reuse in Part IV

Next: **Part IV**, where we put 2018 and 2024 side by side to actually measure what changed.

## *PART IV: Land Use Change*

Two independent looks at the same six years of change on Negros: Hansen's own annual forest-loss labels, and change detected from *our own* homemade land-cover classifier from Part III — cross-checked against each other, the same way Part I cross-checked its homemade climate map against Beck et al.

### 1) Forest cover loss

`negros_forest_change.nc` has a second variable we haven't used yet: `lossyear` — the year each pixel first lost forest cover (1 = 2001, ..., 23 = 2023, 0 = no loss detected). Let's load it:

In [ ]:
lossyear = forest_ds["lossyear"]        # already opened in Part II as forest_ds
lossyear

### 🖊️ How many pixels were lost each year?

`np.unique(..., return_counts=True)` is the numpy equivalent of pandas' `.value_counts()` from Part II, applied to a plain array:

In [ ]:
years, counts = np.unique(..., return_counts=True)     # ✏️ of what do we want the unique values? Hint: the lossyear.values

for year_code, count in zip(years, counts):
    if year_code == 0:
        continue                        # 0 = no loss detected, not an actual year
    print(f"{2000 + int(year_code)}: {count} pixels")

**Bar plot** of pixels lost per year:

In [ ]:
loss_only = counts[years != 0]
years_only = 2000 + years[years != 0].astype(int)

fig, ax = plt.subplots(figsize=(11, 4))

ax.bar(..., ..., color="firebrick")           # ✏️ x: years, y: loss

ax.set_title("Forest Loss per Year — Negros (Hansen et al.)")
ax.set_xlabel("Year")
ax.set_ylabel("Pixels lost (30m)")

> 💬 **Discussion:** Any standout years? *(Real data shows a clear spike around 2018, and again in 2022-2023 — any guesses why, before we tell you? Think El Niño-driven dry seasons/fires, logging booms, or just natural year-to-year noise.)*

Map `lossyear` spatially (masking the "no loss" pixels so they don't dominate the colour scale), next to `treecover2000`:

In [ ]:
lossyear_masked = lossyear.where(lossyear > 0)      # 💡 mask out "no loss" (0) so it doesn't dominate the color scale

fig, axes = plt.subplots(1, 2, figsize=(13, 8))

lossyear_masked.plot(ax=axes[0], cmap="YlOrRd", cbar_kwargs={"label": "Loss year code (1=2001 ... 23=2023)"})
axes[0].set_title("Year of forest loss")

treecover.plot(ax=axes[1], cmap="Greens", vmin=0, vmax=100, add_colorbar=False)
axes[1].set_title("Tree cover (2000)")

for ax in axes:
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Where has loss concentrated? Does it look random, or clustered along certain patterns (roads, forest edges, specific regions)?

### 🖊️ Quantify loss over the 2018-2023 window

This is the same window our two Sentinel-2 snapshots span — setting up the cross-check in subsection 2:

## *PART IV: Land Use Change*

### 1) Forest cover loss
... -> using the satellite imagery from the previous exercises

### 2) Urban expansion
... -> same same